# YOLO11 Training on Google Colab

This notebook trains a YOLO11 model on the 'ok' hand gesture dataset stored in Google Drive.

In [ ]:
# Cell 1: Mount Drive + Install Dependencies
from google.colab import drive
import subprocess
import sys

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Install ultralytics
print("\nInstalling ultralytics...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])

from ultralytics import YOLO
import torch

print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Using CPU (slower)")

In [ ]:
# Cell 2: Configure Paths
from pathlib import Path
import os

# ⚠️ IMPORTANT: Update this path to match your Google Drive structure
# Example: If your data is at MyDrive/Projects/trashrobot/data/processed_from_raw
DRIVE_PATH = "/content/drive/MyDrive/data/processed_from_raw"

# Define paths
DATASET_YAML = Path(DRIVE_PATH) / "dataset.yaml"
IMAGES_TRAIN = Path(DRIVE_PATH) / "images" / "train"
IMAGES_VAL = Path(DRIVE_PATH) / "images" / "val"
LABELS_TRAIN = Path(DRIVE_PATH) / "labels" / "train"
LABELS_VAL = Path(DRIVE_PATH) / "labels" / "val"

# Verify data exists
print("\n📁 Verifying dataset structure...")
print("="*50)
print(f"Dataset YAML exists: {DATASET_YAML.exists()}")
print(f"Images train exists: {IMAGES_TRAIN.exists()}")
print(f"Images val exists: {IMAGES_VAL.exists()}")
print(f"Labels train exists: {LABELS_TRAIN.exists()}")
print(f"Labels val exists: {LABELS_VAL.exists()}")

# Count images and labels
if IMAGES_TRAIN.exists():
    train_images = list(IMAGES_TRAIN.glob('*.jpg'))
    print(f"\n📊 Train images: {len(train_images)}")

if IMAGES_VAL.exists():
    val_images = list(IMAGES_VAL.glob('*.jpg'))
    print(f"📊 Val images: {len(val_images)}")

if LABELS_TRAIN.exists():
    train_labels = list(LABELS_TRAIN.glob('*.txt'))
    print(f"📊 Train labels: {len(train_labels)}")

if LABELS_VAL.exists():
    val_labels = list(LABELS_VAL.glob('*.txt'))
    print(f"📊 Val labels: {len(val_labels)}")

if not all([DATASET_YAML.exists(), IMAGES_TRAIN.exists(), IMAGES_VAL.exists()]):
    print("\n❌ ERROR: Some paths don't exist!")
    print("Please check your DRIVE_PATH and ensure data is uploaded correctly.")

In [ ]:
# Cell 3: Display Sample Image with Annotation
import random
from PIL import Image, ImageDraw
from IPython.display import display

def draw_yolo_boxes(image_path, label_path):
    """Draw YOLO format bounding boxes on image
    YOLO format: [class, x_center, y_center, width, height] (normalized 0-1)
    """
    img = Image.open(image_path)
    draw = ImageDraw.Draw(img)
    width, height = img.size
    
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1])
                    y_center = float(parts[2])
                    w = float(parts[3])
                    h = float(parts[4])
                    
                    # Convert normalized to pixel coordinates
                    x1 = int((x_center - w/2) * width)
                    y1 = int((y_center - h/2) * height)
                    x2 = int((x_center + w/2) * width)
                    y2 = int((y_center + h/2) * height)
                    
                    # Draw rectangle
                    draw.rectangle([x1, y1, x2, y2], outline='red', width=3)
                    label_text = 'ok' if class_id == 0 else 'no_gesture'
                    draw.text((x1, y1-15), label_text, fill='red')
    
    return img

# Pick random validation image
if IMAGES_VAL.exists() and val_images:
    sample_img = random.choice(val_images)
    label_file = LABELS_VAL / (sample_img.stem + '.txt')
    
    print(f"\n📸 Sample: {sample_img.name}")
    print(f"   Size: {Image.open(sample_img).size}")
    print(f"   Label: {label_file.name}")
    
    img = draw_yolo_boxes(sample_img, label_file)
    display(img)

In [ ]:
# Cell 4: Load Model
print("\nLoading YOLO model...")

# Load YOLO11 nano (smallest, fastest - good for Colab free tier)
model = YOLO('yolo11n.pt')

print(f"✅ Model loaded: yolo11n")
print(f"   Task: {model.task}")
print(f"   Classes: {model.names}")

In [ ]:
# Cell 5: Train
print("\nStarting training...")
print("This will take 10-20 minutes on Colab GPU\n")

# Create temporary dataset.yaml pointing to Google Drive
temp_yaml = Path('/content/temp_dataset.yaml')
yaml_content = f"""
path: {DRIVE_PATH}
train: images/train
val: images/val
nc: 2
names: ['ok', 'no_gesture']
"""

with open(temp_yaml, 'w') as f:
    f.write(yaml_content)

print(f"📄 Dataset config: {temp_yaml}")
print(f"\nConfiguration:")
print(yaml_content)

# Determine device
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"\n🖥️  Using device: {device}\n")

# Training configuration
results = model.train(
    data=str(temp_yaml),              # Path to dataset config
    epochs=20,                        # Training iterations
    imgsz=640,                        # Image size
    batch=16,                         # Batch size (reduce to 8 if OOM)
    device=device,                    # '0' for GPU, 'cpu' for CPU
    patience=10,                      # Early stopping patience
    save=True,                        # Save best model
    project='/content/runs/detect',   # Save to local Colab storage (faster)
    name='train',                     # Run name
    exist_ok=True,                    # Overwrite if exists
    verbose=True,                     # Show progress
    # Augmentation for rotation/flip robustness
    degrees=25.0,                     # Random rotation up to ±25 degrees
    flipud=0.5,                       # 50% vertical flip probability
    fliplr=0.5,                       # 50% horizontal flip probability
    shear=10.0,                       # Shear transformation up to ±10 degrees
    perspective=0.001,                # Slight perspective distortion
    mixup=0.1,                        # Mixup augmentation
    copy_paste=0.1                    # Copy-paste augmentation
)
print("\n✅ Training complete!")

In [ ]:
# Cell 6: View Results
from pathlib import Path
from IPython.display import Image as IPImage, display

results_dir = Path('/content/runs/detect/train')

print("\n📊 Training Results")
print("="*50)

# Display results.png
results_img = results_dir / 'results.png'
if results_img.exists():
    print("\n📈 Training Curves:")
    display(IPImage(filename=str(results_img)))

# Display confusion matrix
cm_img = results_dir / 'confusion_matrix.png'
if cm_img.exists():
    print("\n🎯 Confusion Matrix:")
    display(IPImage(filename=str(cm_img)))

# Display validation predictions
val_pred = results_dir / 'val_batch0_pred.jpg'
if val_pred.exists():
    print("\n👁️  Validation Predictions:")
    display(IPImage(filename=str(val_pred)))

# List model weights
weights_dir = results_dir / 'weights'
if weights_dir.exists():
    print("\n💾 Model Weights:")
    for w in weights_dir.glob('*.pt'):
        size_mb = w.stat().st_size / (1024*1024)
        print(f"   - {w.name} ({size_mb:.1f} MB)")

In [ ]:
# Cell 7: Test Inference
import random

print("\n🧪 Testing Trained Model\n")

# Load best model
best_model_path = Path('/content/runs/detect/train/weights/best.pt')

if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    
    # Pick random validation image
    if val_images:
        test_img = random.choice(val_images)
        print(f"Testing on: {test_img.name}\n")
        
        # Run inference
        results = best_model(test_img, verbose=False)
        
        # Display results
        for result in results:
            im_array = result.plot()
            display(Image.fromarray(im_array))
            
            # Print detections
            if result.boxes:
                print(f"\n✅ Detected {len(result.boxes)} object(s):")
                for i, box in enumerate(result.boxes):
                    class_id = int(box.cls)
                    class_name = best_model.names[class_id]
                    conf = float(box.conf)
                    print(f"   {i+1}. {class_name}: {conf:.2%}")
            else:
                print("\n⚠️  No objects detected")
else:
    print("❌ Best model not found. Training may have failed.")

In [ ]:
# Cell 8: Save Results to Drive (Optional)
import shutil

print("\n💾 Saving Results to Google Drive\n")

# Create results folder in Drive (sibling to data folder)
save_dir = Path(DRIVE_PATH).parent / 'training_results'
save_dir.mkdir(exist_ok=True)

print(f"Destination: {save_dir}")
print("="*50)

# Copy files
files_to_copy = [
    ('best.pt', results_dir / 'weights' / 'best.pt'),
    ('last.pt', results_dir / 'weights' / 'last.pt'),
    ('results.png', results_dir / 'results.png'),
    ('confusion_matrix.png', results_dir / 'confusion_matrix.png'),
]

for name, src in files_to_copy:
    if src.exists():
        dst = save_dir / name
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / (1024*1024)
        print(f"✅ {name} ({size_mb:.1f} MB)")
    else:
        print(f"⚠️  {name} not found")

# Copy entire weights folder
weights_src = results_dir / 'weights'
weights_dst = save_dir / 'weights'
if weights_src.exists():
    if weights_dst.exists():
        shutil.rmtree(weights_dst)
    shutil.copytree(weights_src, weights_dst)
    print(f"✅ weights/ folder")

print(f"\n📁 All results saved to: {save_dir}")

In [ ]:
# Cell 9: Export to ONNX (Optional)
print("\n📤 Exporting Model to ONNX\n")

# Load best model
if best_model_path.exists():
    model = YOLO(str(best_model_path))
    
    # Export to ONNX
    print("Exporting to ONNX format...")
    model.export(format='onnx', imgsz=640)
    
    # Copy ONNX to Drive
    onnx_path = results_dir / 'weights' / 'best.onnx'
    if onnx_path.exists():
        dst = save_dir / 'best.onnx'
        shutil.copy2(onnx_path, dst)
        size_mb = dst.stat().st_size / (1024*1024)
        print(f"✅ ONNX model saved: {dst} ({size_mb:.1f} MB)")
    else:
        print("⚠️  ONNX export failed")
else:
    print("❌ Best model not found. Cannot export.")